# 🛰️ Monitoring & Evaluating Agentic AI in Production with Langfuse

**hands-on workshop** — you will build a multi-agent customer-support system for a housing-finance
company, then wrap the full production loop around it: **tracing → prompt management → user feedback →
evaluation (rules, LLM judges, DeepEval) → online evaluation → dashboards → monitoring & alerting → CI/CD gates.**

**The use case** — *Meridian Housing Finance* (fictional; all data is synthetic) runs an AI support desk that
answers
- loan-account questions
- explains policies from a knowledge base
- raises service tickets.
  
We will ship a deliberately imperfect **v1**, catch its failures with evaluation, ship **v2** and prove the
improvement, then catch a bad **v3** deploy in production and roll it back.

> **Meridian AgentOps workshop · notebook 00 of 6 (Modules 0–2).** The reusable code lives in the
> repo's `app/` package (agent, tools, evaluators, config) — these notebooks
> import it, so a fresh session only needs the bootstrap cells below instead of re-running
> earlier modules. **Prerequisite:** none — this is where the workshop starts.


## Module 0 · Setup & connection check

**Before running anything**:
1. Your Langfuse keys: [cloud.langfuse.com](https://cloud.langfuse.com) → your project →
   **Settings ▸ API Keys**; your OpenAI key: [platform.openai.com](https://platform.openai.com/api-keys).
2. Paste them into the keys cell (0.2) below. Colab saves your copy of the notebook to
   Drive, so each notebook keeps your keys across session resets — a one-time edit.
   (Testing keys only — don't share your notebook copy.)


In [ ]:
# ── 0.1 Get the code + the pinned stack (fresh Colab VM: clone first) ──
import os
if not os.path.isdir("../app"):                    # fresh Colab VM → clone the repo
    !git clone https://github.com/kartik-nighania/data-hack-summit-2026.git _workshop_repo
    %cd _workshop_repo/workshop
%pip install -q -r ../requirements.txt
print("✅ stack ready — if pip just upgraded packages, do Run ▸ Restart session once and rerun from the top.")


In [ ]:
import os, sys, re

sys.path.insert(0, os.path.abspath(".."))        # make the repo's app/ package importable

# Jupyter kernels already run an event loop; this lets libraries that call
# asyncio.run()/run_until_complete work inside notebook cells.
import nest_asyncio
nest_asyncio.apply()

print("✅ environment prepared |", sys.version.split()[0])


In [ ]:
# ── 0.2 Your API keys — paste them in, then run. Colab saves your notebook copy to Drive,
#        so this is a one-time edit per notebook (testing keys only — don't share the copy).
import os
os.environ["OPENAI_API_KEY"] = "sk-proj-..."          # platform.openai.com/api-keys
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."       # cloud.langfuse.com → Settings ▸ API Keys
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
os.environ["LANGFUSE_BASE_URL"] = "https://us.cloud.langfuse.com"   # EU account: https://cloud.langfuse.com

from app.config import load_keys
load_keys()   # validates; unedited placeholders fall back to .env or a prompt


In [ ]:
# ── 0.3 Constants (config.yaml) + the Langfuse client (PII masking hook registered) ─
from app.config import AGENT_MODEL, get_lf
from app.pii_data_masking import PII_PATTERNS      # filled in 2.9 below
lf = get_lf()
if not lf.auth_check():
    raise SystemExit("❌ Langfuse authentication FAILED. Check keys + LANGFUSE_HOST region "
                     "(EU: https://cloud.langfuse.com / US: https://us.cloud.langfuse.com).")
LANGFUSE_HOST = os.environ["LANGFUSE_HOST"]
import importlib.metadata as _md
print("✅ Langfuse authenticated:", LANGFUSE_HOST)
for p in ["langfuse", "langchain", "langgraph", "deepeval", "openai"]:
    print(f"   {p}=={_md.version(p)}")


In [ ]:
from openai import OpenAI

from langfuse import observe   # trace_url comes from app.config

@observe(name="connection-check")
def connection_check():
    client = OpenAI()
    resp = client.chat.completions.create(
        model=AGENT_MODEL,
        messages=[{"role": "user", "content": "Reply with exactly: pong"}],
    )
    return resp.choices[0].message.content

print("OpenAI says:", connection_check())
lf.flush() 
print("✅ First trace sent. Open your Langfuse project → Tracing → newest trace ('connection-check').")

## Module 1 · Observability foundations

Three disciplines, three questions:

| Discipline | Question it answers | Langfuse feature |
|---|---|---|
| **Tracing** | *What exactly happened in this request?* | Traces, observations, sessions, users, tags |
| **Monitoring** | *Is the system healthy right now?* | Dashboards, Metrics API, monitors & alerts |
| **Evaluation** | *Is the output any good — and is it getting better?* | Scores, datasets, experiments, judges, annotation |


**How Langfuse organizes data** :
- **Trace** = one request through your system → contains a tree of **observations**
  (`span` = any step, `generation` = an LLM call with tokens/cost, plus typed spans: `agent`, `tool`, `retriever`…).
- **Session** = many traces of one conversation (`session_id` — we'll map it to the LangGraph thread id).
  **User** = `user_id` across traces/sessions → per-user views. **Tags / metadata / environment / version** slice traffic.
- **Score** = a judgment attached to a trace / observation / session / dataset run — NUMERIC, CATEGORICAL,
  BOOLEAN or TEXT — written by rules, LLM judges, humans, or your app's feedback buttons.
- **Prompt** = versioned, labeled, deployable asset.
- **Dataset → Experiment runs** = offline evaluation.

## Module 2 · Build the multi-agent & wire in Langfuse

In [ ]:
# ── 2.1 The mock world: customers, loans, tickets (synthetic data, incl. PII) ─
# now lives in data/database.json + data/knowledge_base.json, loaded by app/db.py
from app.db import CUSTOMERS, LOANS, TICKETS
print(f"Mock world ready: {len(CUSTOMERS)} customers, {len(LOANS)} loans, {len(TICKETS)} ticket(s)")


In [ ]:
# ── 2.2 The policy knowledge base + a tiny in-memory TF-IDF retriever ────────
# lives in app/tools.py, indexing data/knowledge_base.json
from app.tools import retriever
for pid, score, text in retriever.search("What are the charges to foreclose my loan early?"):
    print(pid, score, text)
    print('')


In [ ]:
# Tools (the specialists' hands) ───────────────────────────────────────
# all live in app/tools.py — the account, policy and service-desk specialists' tools
from app.tools import ACCOUNT_TOOLS, POLICY_TOOLS, SERVICE_TOOLS
print("Tools ready:", [t.name for t in ACCOUNT_TOOLS + POLICY_TOOLS + SERVICE_TOOLS])


In [ ]:
# ── 2.4 The three managed prompts (v1 - flawed on purpose) → into Langfuse ───
# texts + idempotent ensure_prompt() live in app/prompts.py (flaws marked "# ← flaw")
from app.prompts import seed_prompts
versions = seed_prompts()
print("Prompts in Langfuse:", ", ".join(f"{k} v{v}" for k, v in versions.items()))
print("→ open Langfuse ▸ Prompts to see them (labels: production, latest)")


In [ ]:
# Assemble the graph: supervisor → specialists → finalize ──────────────
# lives in app/agent.py; deploy_agent() bakes in the current production prompts
from app.agent import get_agent
from IPython.display import Image, display
AGENT_GRAPH = get_agent()
print("Compiled. Topology (mermaid — Langfuse will render this live in the Agent Graph tab):\n")
display(Image(AGENT_GRAPH.get_graph().draw_mermaid_png()))


In [ ]:
# ── run_agent: the production entry point (root span + identity mapping) ─
# run_agent() / invoke_agent() + _package() live in app/agent.py (async ainvoke_agent() powers experiments)
from app.agent import run_agent
r = run_agent("What is my current EMI amount, and which date is it debited every month?",
              "CUST-1001", session_id="sess-m2-hello", tags=["module-2"])
print("ANSWER:\n", r["answer"])
print("\nroute:", r["route"], "| tools:", [t["name"] for t in r["tool_calls"]], f"| {r['latency_s']}s")
lf.flush()
print("\n🔗 open this trace:", r["url"])


### ✅ CHECKPOINT — your first agent trace
- full tree: `meridian-support-agent` (root) → `supervisor`
(structured-output generation) → `account_agent` → tool span `get_loan_summary` → back to `supervisor` →
`finalize`.
- Note **tokens and cost on every generation**, and open the **Agent Graph** tab — Langfuse renders
the LangGraph topology automatically.
- The supervisor generation shows **`support-supervisor v1`** linked under
*Prompt* — that link powers per-version metrics in Module 3.

In [ ]:
# ── 2.8 The masking problem: PII is flowing into your observability stack ────
r = run_agent("Please confirm which phone number and PAN you have on file for me.",
              "CUST-1001", session_id="sess-m2-pii", tags=["module-2", "pii-demo"])
print(r["answer"])
lf.flush()
print("\n⚠️  Open this trace and look at the get_customer tool output:", r["url"])
print("    Ananya's real phone number and PAN are sitting in your Langfuse project.")

In [ ]:
# ── 2.9 Fix: register PII patterns → every span is redacted AT EXPORT TIME ───
# (The hook was registered on the client in Module 0; until now its pattern list was empty.)
PII_PATTERNS.extend([
    (re.compile(r"\b[A-Z]{5}\d{4}[A-Z]\b"), "«PAN-REDACTED»"),                     # Indian PAN
    (re.compile(r"\+91[-\s]?\d{10}\b"), "«PHONE-REDACTED»"),                      # +91 phone numbers
    (re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.]+\b"), "«EMAIL-REDACTED»"),           # emails
])

r = run_agent("Please confirm which phone number and PAN you have on file for me.",
              "CUST-1001", session_id="sess-m2-pii", tags=["module-2", "pii-demo", "masked"])
print(r["answer"][:400])
lf.flush()
print("\n🔒 Same question, new trace:", r["url"])
print("   Tool outputs and the answer now show «PHONE-REDACTED» / «PAN-REDACTED» in Langfuse —")
print("   the model still saw the real values; only the EXPORTED telemetry is masked.")

### ✅ CHECKPOINT — masking
- Compare the two `pii-demo` traces: the first leaks the phone/PAN, the second shows `«…-REDACTED»` everywhere —
- `mask_otel_spans` patches the raw OTel attributes at export.